# TP3: Image Segmentation with U-Net


**Instructor:** Vivien Conti

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sorbonnescai/2026_bootcamp_spatial/blob/main/warmups/tp3_sementation.ipynb)

---

## Objectives

In this practical, you will learn **image segmentation** - the task of classifying every pixel in an image to delineate precise object boundaries.

By the end of this practical, you will:

1. **Understand segmentation**: From bounding boxes to pixel-precise masks
2. **Learn mask format**: How segmentation labels work at the pixel level
3. **Fine-tune U-Net**: Adapt a pre-trained encoder-decoder to segment cell nuclei
4. **Evaluate segmentation**: Understand IoU (Jaccard) and Dice score

---

# Part 1: Segmentation vs Detection vs Classification

## The Computer Vision Task Hierarchy

| Task | Input | Output | Granularity |
|------|-------|--------|-------------|
| **Classification** (TP1) | Image | Single label | Image-level |
| **Detection** (TP2) | Image | Bounding boxes + labels | Object-level |
| **Segmentation** (TP3) | Image | Pixel-wise mask | Pixel-level |

```
Classification:          Detection:              Segmentation:
+-----------+           +-----------+           +-----------+
|           |           | [box]     |           | ####      |
|  (image)  | -> "cell" | [box]     |           | ####      |
|           |           |           |           |      #### |
+-----------+           +-----------+           +-----------+
 One label              Boxes + labels           Every pixel labeled!
```

## Why Segmentation?

In many scientific applications, bounding boxes are not enough:
- **Cell biology**: Measure cell area, shape, and morphology
- **Medical imaging**: Delineate tumors for treatment planning
- **Remote sensing**: Map land use at pixel resolution
- **Materials science**: Identify grain boundaries and defects

## U-Net: The Standard for Scientific Segmentation

**U-Net** (Ronneberger et al., 2015) was invented for biomedical image segmentation. Its encoder-decoder architecture with **skip connections** preserves fine spatial details:

```
U-Net Architecture:

  Encoder (ResNet18)              Decoder
  ┌──────────┐                  ┌──────────┐
  │ 256x256  │ ──skip connect──>│ 256x256  │ -> Mask
  │ |  pool  │                  │ ^ upsamp │
  │ 128x128  │ ──skip connect──>│ 128x128  │
  │ |  pool  │                  │ ^ upsamp │
  │  64x64   │ ──skip connect──>│  64x64   │
  │ |  pool  │                  │ ^ upsamp │
  │  32x32   │ ──skip connect──>│  32x32   │
  │ |  pool  │                  │ ^ upsamp │
  └──── Bottleneck (16x16) ─────┘          │
         compress then expand               │
```

- **Encoder**: Pre-trained CNN (e.g. ResNet18) extracts features at multiple scales
- **Decoder**: Upsamples back to original resolution
- **Skip connections**: Pass high-resolution details from encoder to decoder

## Setup

In [ ]:
# Install required packages
!pip install -q segmentation-models-pytorch albumentations

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from PIL import Image
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("Setup complete!")

---

# Part 2: The Cell Nuclei Dataset

We'll use a **cell nuclei segmentation dataset** - microscopy images of cells with labeled nucleus boundaries.

This is a **binary segmentation** task:
- **Class 0 (background)**: Everything that is not a nucleus
- **Class 1 (nucleus)**: Cell nuclei

This is directly relevant to biomedical research:
- Counting cells in tissue samples
- Measuring nucleus morphology
- Tracking cell division

In [ ]:
# Download and extract the dataset
!wget -q https://www.raphaelcousin.com/modules/sandbox/cell_nuclei_segmentation.zip
!unzip -q -o cell_nuclei_segmentation.zip
!rm cell_nuclei_segmentation.zip

# Check the structure
print("Dataset structure:")
!ls -la

In [ ]:
# Count images in each split
train_images = sorted(Path('train/images').glob('*.png'))
train_masks = sorted(Path('train/masks').glob('*.png'))
valid_images = sorted(Path('valid/images').glob('*.png'))
valid_masks = sorted(Path('valid/masks').glob('*.png'))
test_images = sorted(Path('test/images').glob('*.png'))
test_masks = sorted(Path('test/masks').glob('*.png'))

print(f"Dataset splits:")
print(f"  Train:      {len(train_images)} images, {len(train_masks)} masks")
print(f"  Validation: {len(valid_images)} images, {len(valid_masks)} masks")
print(f"  Test:       {len(test_images)} images, {len(test_masks)} masks")
print(f"  Total:      {len(train_images) + len(valid_images) + len(test_images)} images")

## Understanding Segmentation Masks

Unlike detection (bounding boxes), segmentation uses **pixel-level labels**:

```
Image (RGB):                    Mask (binary):
┌─────────────────┐            ┌─────────────────┐
│  pixel values   │            │ 0 0 0 0 0 0 0 0 │
│  0-255 per      │            │ 0 0 1 1 1 0 0 0 │
│  channel (R,G,B)│     ->     │ 0 1 1 1 1 1 0 0 │
│                 │            │ 0 0 1 1 1 0 0 0 │
│                 │            │ 0 0 0 0 0 1 1 0 │
│                 │            │ 0 0 0 0 0 1 1 0 │
└─────────────────┘            └─────────────────┘
                                0 = background
                                1 = nucleus
```

Each pixel in the mask is either 0 (background) or 1 (nucleus).

In [ ]:
# Visualize sample images with their masks
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

sample_indices = random.sample(range(len(train_images)), 3)

for row, idx in enumerate(sample_indices):
    img = np.array(Image.open(train_images[idx]))
    mask = np.array(Image.open(train_masks[idx]).convert('L'))
    mask_binary = (mask > 127).astype(np.uint8)

    # Original image
    axes[row, 0].imshow(img)
    axes[row, 0].set_title('Image', fontsize=11)
    axes[row, 0].axis('off')

    # Mask
    axes[row, 1].imshow(mask_binary, cmap='gray')
    axes[row, 1].set_title('Mask', fontsize=11)
    axes[row, 1].axis('off')

    # Overlay
    axes[row, 2].imshow(img)
    axes[row, 2].imshow(mask_binary, alpha=0.4, cmap='Reds')
    axes[row, 2].set_title('Overlay', fontsize=11)
    axes[row, 2].axis('off')

plt.suptitle('Cell Nuclei Dataset - Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze pixel class balance
fg_ratios = []
for mask_path in train_masks[:50]:  # Sample 50 masks
    mask = np.array(Image.open(mask_path).convert('L'))
    fg_ratio = (mask > 127).mean()
    fg_ratios.append(fg_ratio)

print(f"Pixel-level class balance (sampled from training set):")
print(f"  Foreground (nucleus): {np.mean(fg_ratios)*100:.1f}% of pixels")
print(f"  Background:           {(1-np.mean(fg_ratios))*100:.1f}% of pixels")
print(f"  Imbalance ratio:      1:{1/np.mean(fg_ratios):.0f}")
print(f"\n  -> The dataset is heavily imbalanced! Standard cross-entropy")
print(f"     would bias toward predicting everything as background.")

### Question 1

Look at the sample images and masks:

1. How many nuclei are typically in each image?
2. What percentage of pixels belong to nuclei vs background? Why is this a problem?
3. What challenges do you notice? (touching nuclei, varying sizes, blurry boundaries?)

---

# Part 3: Data Pipeline

For segmentation, we need to apply **the same spatial transforms** to both image and mask simultaneously.

We use **albumentations** instead of torchvision transforms because it natively handles paired image+mask augmentation.

```
  Image           Mask
  ┌───┐          ┌───┐
  │ A │ rotate   │ A │  rotate
  └───┘  45°     └───┘   45°
    |              |       <- Same transform!
    v              v
  ┌───┐          ┌───┐
  │/A/│          │/A/│
  └───┘          └───┘
```

In [ ]:
# Define transforms
IMG_SIZE = 256

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

print("Transforms defined!")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Train augmentations: flip, rotate, color jitter")
print(f"  Val/Test: resize + normalize only")

In [ ]:
class CellDataset(Dataset):
    """Custom dataset for cell nuclei segmentation."""

    def __init__(self, image_dir, mask_dir, transform=None):
        self.images = sorted(Path(image_dir).glob('*.png'))
        self.masks = sorted(Path(mask_dir).glob('*.png'))
        self.transform = transform
        assert len(self.images) == len(self.masks), "Mismatch between images and masks!"

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image and mask
        image = np.array(Image.open(self.images[idx]).convert('RGB'))
        mask = np.array(Image.open(self.masks[idx]).convert('L'))
        mask = (mask > 127).astype(np.float32)  # Binary: 0 or 1

        # Apply transforms (same spatial transform to both!)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.unsqueeze(0).float()  # mask shape: (1, H, W)


# Create datasets
train_dataset = CellDataset('train/images', 'train/masks', transform=train_transform)
val_dataset = CellDataset('valid/images', 'valid/masks', transform=val_transform)
test_dataset = CellDataset('test/images', 'test/masks', transform=val_transform)

print(f"Datasets created:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")

# Check shapes
img, msk = train_dataset[0]
print(f"\nSample shapes:")
print(f"  Image: {img.shape} (C, H, W)")
print(f"  Mask:  {msk.shape} (1, H, W)")

In [ ]:
# Visualize augmented samples
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Augmented Training Samples', fontsize=14, fontweight='bold')

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(4):
    img, mask = train_dataset[random.randint(0, len(train_dataset)-1)]

    # Denormalize image for display
    img_display = img.permute(1, 2, 0).numpy() * std + mean
    img_display = np.clip(img_display, 0, 1)

    axes[0, i].imshow(img_display)
    axes[0, i].set_title('Image', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(mask.squeeze(), cmap='gray')
    axes[1, i].set_title('Mask', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Create data loaders
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Data loaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

---

# Part 4: Segmentation Metrics

For segmentation, accuracy alone is misleading (predicting all background = 90%+ accuracy!).

We use **IoU** (Intersection over Union) and **Dice** score:

```
IoU (Jaccard Index):

 Predicted:     Ground Truth:    Intersection:    Union:
  ####            ####              ###            #####
  ####            ####              ###            #####
                  ####                             ####

 IoU = |Intersection| / |Union|

 Dice = 2 * |Intersection| / (|Predicted| + |Ground Truth|)
```

| Metric | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| **IoU** | Intersection / Union | 0-1 | 0.5+ = decent, 0.7+ = good |
| **Dice** | 2 * Intersection / (Pred + GT) | 0-1 | Always >= IoU |

In [ ]:
def compute_iou(pred, target, threshold=0.5):
    """Compute IoU (Intersection over Union) for binary segmentation."""
    pred_binary = (pred > threshold).float()
    intersection = (pred_binary * target).sum()
    union = pred_binary.sum() + target.sum() - intersection
    return (intersection + 1e-8) / (union + 1e-8)  # Smooth to avoid 0/0


def compute_dice(pred, target, threshold=0.5):
    """Compute Dice coefficient for binary segmentation."""
    pred_binary = (pred > threshold).float()
    intersection = (pred_binary * target).sum()
    return (2 * intersection + 1e-8) / (pred_binary.sum() + target.sum() + 1e-8)


# Quick test with perfect and random predictions
dummy_target = torch.zeros(1, 1, 8, 8)
dummy_target[0, 0, 2:6, 2:6] = 1  # Square nucleus

print("Metric sanity check:")
print(f"  Perfect prediction:  IoU={compute_iou(dummy_target, dummy_target):.3f}, Dice={compute_dice(dummy_target, dummy_target):.3f}")
print(f"  Random prediction:   IoU={compute_iou(torch.rand_like(dummy_target), dummy_target):.3f}, Dice={compute_dice(torch.rand_like(dummy_target), dummy_target):.3f}")
print(f"  All zeros (no pred): IoU={compute_iou(torch.zeros_like(dummy_target), dummy_target):.3f}, Dice={compute_dice(torch.zeros_like(dummy_target), dummy_target):.3f}")

---

# Part 5: Pre-trained U-Net - Before Fine-tuning

Let's load a U-Net with a pre-trained ResNet18 encoder and see what it predicts **before fine-tuning**.

The encoder is pre-trained on ImageNet, but the decoder has random weights.

In [ ]:
# Load U-Net with pre-trained encoder
model_pretrained = smp.Unet(
    encoder_name='resnet18',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,  # Binary segmentation: 1 output channel
)

total_params = sum(p.numel() for p in model_pretrained.parameters())
encoder_params = sum(p.numel() for p in model_pretrained.encoder.parameters())
decoder_params = total_params - encoder_params

print(f"U-Net with ResNet18 encoder")
print(f"  Total parameters:   {total_params:,}")
print(f"  Encoder parameters: {encoder_params:,} (pre-trained on ImageNet)")
print(f"  Decoder parameters: {decoder_params:,} (random weights)")

In [ ]:
# Test pre-trained model on sample images (before fine-tuning)
model_pretrained.eval()
model_pretrained.to(device)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for row in range(3):
    img, mask = val_dataset[random.randint(0, len(val_dataset)-1)]

    with torch.no_grad():
        pred = model_pretrained(img.unsqueeze(0).to(device))
        pred = torch.sigmoid(pred).cpu().squeeze()

    # Display
    img_display = img.permute(1, 2, 0).numpy() * std + mean
    img_display = np.clip(img_display, 0, 1)

    axes[row, 0].imshow(img_display)
    axes[row, 0].set_title('Image', fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(mask.squeeze(), cmap='gray')
    axes[row, 1].set_title('Ground Truth', fontsize=11)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred.numpy(), cmap='gray', vmin=0, vmax=1)
    axes[row, 2].set_title('Pre-trained Prediction', fontsize=11)
    axes[row, 2].axis('off')

plt.suptitle('U-Net Predictions BEFORE Fine-tuning (random decoder)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Question 2

1. What does the pre-trained model predict? Why is it essentially noise?
2. Which part of the model is pre-trained (encoder) and which is random (decoder)?
3. Why can't we use the pre-trained model directly, unlike in TP1 where a frozen backbone already gave decent results?

---

# Part 6: Fine-tune U-Net

Now we'll fine-tune the full model on our cell nuclei dataset.

**Loss function**: We combine **Dice Loss** and **BCE Loss**:
- **BCE** (Binary Cross-Entropy): Standard pixel-wise classification loss
- **Dice Loss**: Directly optimizes the Dice score, handles class imbalance

Using both gives better results than either alone.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    running_iou = 0.0
    n_batches = 0

    for images, masks in tqdm(loader, desc='Training', leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_iou += compute_iou(torch.sigmoid(outputs), masks).item()
        n_batches += 1

    return running_loss / n_batches, running_iou / n_batches


def evaluate(model, loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    running_loss = 0.0
    running_iou = 0.0
    running_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            preds = torch.sigmoid(outputs)
            running_loss += loss.item()
            running_iou += compute_iou(preds, masks).item()
            running_dice += compute_dice(preds, masks).item()
            n_batches += 1

    return running_loss / n_batches, running_iou / n_batches, running_dice / n_batches


print("Training utilities defined!")

In [ ]:
# Create model for fine-tuning
model = smp.Unet(
    encoder_name='resnet18',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
)
model = model.to(device)

# Loss: Dice + BCE combination
dice_loss = smp.losses.DiceLoss(mode='binary')
bce_loss = nn.BCEWithLogitsLoss()
criterion = lambda pred, target: dice_loss(pred, target) + bce_loss(pred, target)

# Optimizer and scheduler
EPOCHS = 25
LR = 1e-4

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Training configuration:")
print(f"  Model: U-Net (ResNet18 encoder)")
print(f"  Loss: DiceLoss + BCEWithLogitsLoss")
print(f"  Optimizer: Adam (lr={LR})")
print(f"  Scheduler: CosineAnnealing")
print(f"  Epochs: {EPOCHS}")

In [ ]:
# Training loop
history = {'train_loss': [], 'train_iou': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}
best_iou = 0

print(f"\nTraining U-Net")
print("=" * 70)

for epoch in range(EPOCHS):
    train_loss, train_iou = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_iou, val_dice = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)

    # Save best model
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'best_model.pth')

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}: "
              f"Train Loss={train_loss:.4f}, Train IoU={train_iou:.3f}, "
              f"Val Loss={val_loss:.4f}, Val IoU={val_iou:.3f}, Val Dice={val_dice:.3f}")

print(f"\nBest Validation IoU: {best_iou:.3f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', linewidth=2)
ax1.plot(history['val_loss'], label='Validation', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_iou'], label='Train IoU', linewidth=2)
ax2.plot(history['val_iou'], label='Val IoU', linewidth=2)
ax2.plot(history['val_dice'], label='Val Dice', linewidth=2, linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('IoU & Dice Score', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Question 3

Look at the training curves:

1. Is the model **overfitting**? (Gap between train and validation metrics)
2. Has the model **converged**? (Metrics plateaued)
3. Why is Dice always higher than IoU for the same predictions?
4. Compare these curves with TP1 - what similarities and differences do you observe?

---

# Part 7: Evaluate the Fine-tuned Model

Let's visualize predictions and compare with ground truth.

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

# Evaluate on test set
test_loss, test_iou, test_dice = evaluate(model, test_loader, criterion, device)

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  Loss: {test_loss:.4f}")
print(f"  IoU:  {test_iou:.3f}")
print(f"  Dice: {test_dice:.3f}")

In [ ]:
# Visualize predictions on test set
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
columns = ['Image', 'Ground Truth', 'Prediction', 'Overlay']

for col, title in enumerate(columns):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for row in range(4):
    idx = random.randint(0, len(test_dataset) - 1)
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
        pred = torch.sigmoid(pred).cpu().squeeze().numpy()

    pred_binary = (pred > 0.5).astype(np.float32)
    img_display = img.permute(1, 2, 0).numpy() * std + mean
    img_display = np.clip(img_display, 0, 1)
    mask_np = mask.squeeze().numpy()

    # Image
    axes[row, 0].imshow(img_display)
    axes[row, 0].axis('off')

    # Ground truth
    axes[row, 1].imshow(mask_np, cmap='gray')
    axes[row, 1].axis('off')

    # Prediction
    axes[row, 2].imshow(pred_binary, cmap='gray')
    axes[row, 2].axis('off')

    # Overlay: green = correct, red = false positive, blue = false negative
    overlay = np.zeros((*mask_np.shape, 3))
    overlay[..., 1] = (pred_binary * mask_np)             # True positive = green
    overlay[..., 0] = (pred_binary * (1 - mask_np))       # False positive = red
    overlay[..., 2] = ((1 - pred_binary) * mask_np)       # False negative = blue
    axes[row, 3].imshow(img_display)
    axes[row, 3].imshow(overlay, alpha=0.5)
    iou = compute_iou(torch.tensor(pred), torch.tensor(mask_np)).item()
    axes[row, 3].set_xlabel(f'IoU: {iou:.3f}', fontsize=10)
    axes[row, 3].axis('off')

plt.suptitle('Fine-tuned U-Net - Test Set Predictions\n(Green=TP, Red=FP, Blue=FN)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Question 4

Look at the predictions:

1. Where does the model make **false positive** errors (red)? False negatives (blue)?
2. Does the model struggle with **touching nuclei**, small nuclei, or blurry boundaries?
3. How do the results compare to what you could achieve with bounding boxes (TP2)?

---

# Part 8: Frozen vs Fine-tuned Encoder

Like in TP1, let's compare **freezing the encoder** vs **fine-tuning everything**.

```
Frozen encoder:                    Full fine-tuning:
┌─────────────┐                   ┌─────────────┐
│  Encoder    │ FROZEN            │  Encoder    │ TRAINED (small LR)
│  (ResNet18) │ (no gradients)    │  (ResNet18) │ (adapts features)
├─────────────┤                   ├─────────────┤
│  Decoder    │ TRAINED           │  Decoder    │ TRAINED
│  (U-Net)    │                   │  (U-Net)    │
└─────────────┘                   └─────────────┘
  Faster, less data needed          Better performance
```

In [ ]:
# Train a model with FROZEN encoder
model_frozen = smp.Unet(
    encoder_name='resnet18',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
)

# Freeze encoder parameters
for param in model_frozen.encoder.parameters():
    param.requires_grad = False

model_frozen = model_frozen.to(device)

total_params = sum(p.numel() for p in model_frozen.parameters())
trainable_params = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)

print(f"U-Net with FROZEN encoder:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters:    {total_params - trainable_params:,}")
print(f"  Trainable ratio:      {100*trainable_params/total_params:.1f}%")

In [ ]:
# Train frozen model
optimizer_frozen = optim.Adam(filter(lambda p: p.requires_grad, model_frozen.parameters()), lr=1e-3)
scheduler_frozen = optim.lr_scheduler.CosineAnnealingLR(optimizer_frozen, T_max=EPOCHS)

history_frozen = {'train_loss': [], 'train_iou': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}
best_iou_frozen = 0

print(f"\nTraining U-Net (Frozen Encoder)")
print("=" * 70)

for epoch in range(EPOCHS):
    train_loss, train_iou = train_one_epoch(model_frozen, train_loader, criterion, optimizer_frozen, device)
    val_loss, val_iou, val_dice = evaluate(model_frozen, val_loader, criterion, device)
    scheduler_frozen.step()

    history_frozen['train_loss'].append(train_loss)
    history_frozen['train_iou'].append(train_iou)
    history_frozen['val_loss'].append(val_loss)
    history_frozen['val_iou'].append(val_iou)
    history_frozen['val_dice'].append(val_dice)

    if val_iou > best_iou_frozen:
        best_iou_frozen = val_iou

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}: "
              f"Train Loss={train_loss:.4f}, Train IoU={train_iou:.3f}, "
              f"Val Loss={val_loss:.4f}, Val IoU={val_iou:.3f}, Val Dice={val_dice:.3f}")

print(f"\nBest Validation IoU (Frozen): {best_iou_frozen:.3f}")

In [ ]:
# Compare frozen vs fine-tuned
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['val_loss'], label='Fine-tuned', linewidth=2, color='#2ca02c')
ax1.plot(history_frozen['val_loss'], label='Frozen encoder', linewidth=2, color='#ff7f0e')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Validation Loss')
ax1.set_title('Validation Loss Comparison', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['val_iou'], label='Fine-tuned', linewidth=2, color='#2ca02c')
ax2.plot(history_frozen['val_iou'], label='Frozen encoder', linewidth=2, color='#ff7f0e')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation IoU')
ax2.set_title('Validation IoU Comparison', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nComparison:")
print(f"  Fine-tuned best IoU:     {best_iou:.3f}")
print(f"  Frozen encoder best IoU: {best_iou_frozen:.3f}")

### Question 5

1. How does freezing the encoder affect segmentation quality?
2. Is the difference larger or smaller than what you observed in TP1 for classification? Why?
3. When would freezing the encoder be a good strategy? (Hint: think about dataset size)

---

# Part 9: Exercise - Experiment with Architecture

Try modifying the model to improve results.

In [ ]:
# TODO: Experiment with different configurations
# Try modifying these parameters:

# Option 1: Use a larger encoder
# model_exp = smp.Unet(
#     encoder_name='resnet34',  # <-- Try 'resnet34', 'efficientnet-b0', 'mobilenet_v2'
#     encoder_weights='imagenet',
#     in_channels=3,
#     classes=1,
# )

# Option 2: Use a different architecture
# model_exp = smp.DeepLabV3Plus(  # <-- Try DeepLabV3+, FPN, or LinkNet
#     encoder_name='resnet18',
#     encoder_weights='imagenet',
#     in_channels=3,
#     classes=1,
# )

# Option 3: Different loss function
# criterion_exp = smp.losses.FocalLoss(mode='binary')  # Better for imbalanced data

# Option 4: Different image size
# IMG_SIZE = 512  # Higher resolution (slower but more detail)

# Uncomment and train to experiment:
# model_exp = model_exp.to(device)
# optimizer_exp = optim.Adam(model_exp.parameters(), lr=1e-4)
# ...

print("Available encoders in smp:")
print("  resnet18, resnet34, resnet50")
print("  efficientnet-b0, efficientnet-b1")
print("  mobilenet_v2")
print("\nAvailable architectures:")
print("  smp.Unet, smp.UnetPlusPlus, smp.DeepLabV3Plus, smp.FPN, smp.LinkNet")

### Question 6

1. What is the trade-off between encoder size and segmentation quality?
2. How does the architecture choice (U-Net vs DeepLabV3+) affect results?
3. For a real scientific application, how would you decide which model to use?

---

# Summary

## The Computer Vision Pipeline - TP1 to TP3

| TP | Task | Model | Output | Key Metric | Library |
|----|------|-------|--------|------------|--------|
| **TP1** | Classification | ResNet18 | Class label | Accuracy | torchvision |
| **TP2** | Detection | YOLOv8 | Bounding boxes | mAP | ultralytics |
| **TP3** | Segmentation | U-Net (ResNet18) | Pixel mask | IoU / Dice | smp |

## Key Takeaways

1. **Segmentation gives pixel-level precision** - essential when shape and area matter
2. **U-Net with pre-trained encoder** is the standard approach for scientific imaging
3. **Dice + BCE loss** handles class imbalance better than BCE alone
4. **Transfer learning applies to segmentation** too - pre-trained encoders help even when the domain (microscopy) differs from ImageNet
5. **albumentations** is essential for applying consistent transforms to image-mask pairs

---

## Reflection Questions

Before finishing, think about:

1. **In your research**: What structures would you want to segment? (cells, tissues, geological features, satellite imagery?)

2. **Binary vs multi-class**: Would you need binary segmentation (one class) or multi-class (multiple types of structures)?

3. **Annotation tools**: How would you create segmentation masks for your data? (LabelMe, QuPath, napari, CVAT)

4. **Quantitative analysis**: How could segmentation help you measure things in your data? (counting objects, measuring areas, tracking changes over time)